In [ ]:
%pip install -q -e .. matplotlib ipywidgets

# 02 — Subset and plot

Pick a region of interest, fetch only that subset from the EDR server,
and plot it.  This is the workflow most users actually want.

## 1. Discover collections

In [ ]:
import httpx

server = "https://edr.example.com"  # replace with your EDR server root

collections = httpx.get(f"{server}/collections").raise_for_status().json()["collections"]
for c in collections:
    print(c["id"], "-", c.get("title", ""))

In [ ]:
collection_id = collections[0]["id"]  # or pick any id from the list above
collection_url = f"{server}/collections/{collection_id}"

## 2. Open with a bounding box

We narrow the request to Spain.  Only this region is fetched from the
server when we later read `.values`.

In [ ]:
import xarray as xr

import edr_xarray  # registers engine="edr"

spain = (-9.5, 36.0, 3.3, 43.8)  # (lon_min, lat_min, lon_max, lat_max)

ds = xr.open_dataset(collection_url, engine="edr", bbox=spain)
ds

## 3. Pick a date range

The collection covers a range of time.  Inspect it, then pick a
from/to window to animate.

In [ ]:
print("from:", ds.t.values[0])
print("to:  ", ds.t.values[-1])

## 4. Animate with a slider

Set `date_from`/`date_to` to any window, fetch it once with
`.load()`, then drag the slider to scrub through timesteps.
All frames are cached locally — no extra network calls.

In [ ]:
from ipywidgets import interact, IntSlider
import matplotlib.pyplot as plt

date_from = "2024-08-01"  # CHANGE ME — start of window
date_to   = "2024-08-15"  # CHANGE ME — end of window (inclusive)

var = next(iter(ds.data_vars))
frames = ds[var].sel(t=slice(date_from, date_to)).load()

def show(i):
    fig, ax = plt.subplots(figsize=(10, 6))
    frames.isel(t=i).plot(ax=ax, cmap="viridis")
    ax.set_title(f"{var} at {frames.t.values[i]}")
    plt.show()

interact(show, i=IntSlider(min=0, max=len(frames.t) - 1, step=1, value=0))

## 5. Cleanup

In [ ]:
ds.close()